# Titanic - Machine Learning from Disaster parte 2
## Vamos utilizar os dados disponíveis no Kaggle: https://www.kaggle.com/competitions/titanic
## - É um dataset de competição
## - O resultado é avaliado através da acurácia

## Refazendo todo o processo do arquivo analise_titanic4

In [2]:
# Importando o pandas
import pandas as pd

# base de treino
treino = pd.read_csv('train.csv')
treino.head(3)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [3]:
# base de teste
teste = pd.read_csv('test.csv')
teste.head(3)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q


### Tratamento de dados

In [4]:
# Eliminando as colunas com elevada cardinalidade
treino = treino.drop(['Name','Ticket','Cabin'],axis=1)

# Usando a média para substituir valores nulos na coluna de idade
treino.loc[treino.Age.isnull(),'Age'] = treino.Age.mean()

# Tratando a coluna Embarked da base de treino usando a moda 
treino.loc[treino.Embarked.isnull(),'Embarked'] = treino.Embarked.mode()[0]
treino

,PassengerId,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,1,0,3,male,22.000000,1,0,7.2500,S
1,2,1,1,female,38.000000,1,0,71.2833,C
2,3,1,3,female,26.000000,0,0,7.9250,S
3,4,1,1,female,35.000000,1,0,53.1000,S
4,5,0,3,male,35.000000,0,0,8.0500,S
...,...,...,...,...,...,...,...,...,...
886,887,0,2,male,27.000000,0,0,13.0000,S
887,888,1,1,female,19.000000,0,0,30.0000,S
888,889,0,3,female,29.699118,1,2,23.4500,S
889,890,1,1,male,26.000000,0,0,30.0000,C


In [5]:
# Eliminando as colunas com elevada cardinalidade
teste = teste.drop(['Name','Ticket','Cabin'],axis=1)

# Usando a média para substituir valores nulos na coluna de idade
teste.loc[teste.Age.isnull(),'Age'] = teste.Age.mean()

# coluna Fare da base de teste usando a média
teste.loc[teste.Fare.isnull(),'Fare'] = teste.Fare.mean()
teste

,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,892,3,male,34.50000,0,0,7.8292,Q
1,893,3,female,47.00000,1,0,7.0000,S
2,894,2,male,62.00000,0,0,9.6875,Q
3,895,3,male,27.00000,0,0,8.6625,S
4,896,3,female,22.00000,1,1,12.2875,S
...,...,...,...,...,...,...,...,...
413,1305,3,male,30.27259,0,0,8.0500,S
414,1306,1,female,39.00000,0,0,108.9000,C
415,1307,3,male,38.50000,0,0,7.2500,S
416,1308,3,male,30.27259,0,0,8.0500,S


In [6]:
# lambda function para tratar a coluna "Sex"
treino['MaleCheck'] = treino.Sex.apply(lambda x: 1 if x == 'male' else 0)
teste['MaleCheck'] = teste.Sex.apply(lambda x: 1 if x == 'male' else 0)

In [8]:
# Fazendo o RobustScaler das colunas Age e Fare
from sklearn.preprocessing import RobustScaler
transformer = RobustScaler().fit(treino[['Age','Fare']])
treino[['Age','Fare']] = transformer.transform(treino[['Age','Fare']])

# base de teste
transformer = RobustScaler().fit(teste[['Age','Fare']])
teste[['Age','Fare']] = transformer.transform(teste[['Age','Fare']])

In [9]:
# Adicionando a coluna sozinho
def sozinho(a,b):
    if (a == 0 and b == 0):
        return 1
    else:
        return 0
    
treino['Alone'] = treino.apply(lambda x: sozinho(x.SibSp,x.Parch),axis=1)
teste['Alone'] = teste.apply(lambda x: sozinho(x.SibSp,x.Parch),axis=1)

In [11]:
# criando a coluna de familiares
treino['Fam_memb'] = treino.SibSp + treino.Parch
teste['Fam_memb'] = treino.SibSp + treino.Parch

In [12]:
# Fazendo o OrdinalEncoder para a coluna Embarked
from sklearn.preprocessing import OrdinalEncoder
categorias = ['S','C','Q']

enc = OrdinalEncoder(categories=[categorias],dtype='int32')
enc = enc.fit(treino[['Embarked']])
treino['Embarked'] = enc.transform(treino[['Embarked']])

teste['Embarked'] = enc.transform(teste[['Embarked']])

In [13]:
# Apagando as colunas de texto
treino = treino.drop('Sex',axis=1)
teste = teste.drop('Sex',axis=1)

## Visualização das bases após tratamento

In [14]:
treino

,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare,Embarked,MaleCheck,Alone,Fam_memb
0,1,0,3,-0.592240,1,0,-0.312011,0,1,0,1
1,2,1,1,0.638529,1,0,2.461242,1,0,0,1
2,3,1,3,-0.284548,0,0,-0.282777,0,0,1,0
3,4,1,1,0.407760,1,0,1.673732,0,0,0,1
4,5,0,3,0.407760,0,0,-0.277363,0,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,-0.207624,0,0,-0.062981,0,1,1,0
887,888,1,1,-0.823009,0,0,0.673281,0,0,1,0
888,889,0,3,0.000000,1,2,0.389604,0,0,0,3
889,890,1,1,-0.284548,0,0,0.673281,1,1,1,0


In [15]:
teste

,PassengerId,Pclass,Age,SibSp,Parch,Fare,Embarked,MaleCheck,Alone,Fam_memb
0,892,3,0.331562,0,0,-0.280670,2,1,1,1
1,893,3,1.311954,1,0,-0.315800,0,0,0,1
2,894,2,2.488424,0,0,-0.201943,2,1,1,0
3,895,3,-0.256674,0,0,-0.245367,0,1,1,1
4,896,3,-0.648831,1,1,-0.091793,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
413,1305,3,0.000000,0,0,-0.271316,0,1,1,0
414,1306,1,0.684503,0,0,4.001229,1,0,1,0
415,1307,3,0.645287,0,0,-0.305208,0,1,1,0
416,1308,3,0.000000,0,0,-0.271316,0,1,1,2


## Utilizando novos modelos de previsão

### Primeiramente, é necessário separar em treino e validação

In [24]:
from sklearn.model_selection import train_test_split

# Separando a base de treino em X e y
X = treino.drop(['PassengerId','Survived'],axis=1)
Y = treino.Survived

# Separando em treino e validação
X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.33, random_state=42)
X

,Pclass,Age,SibSp,Parch,Fare,Embarked,MaleCheck,Alone,Fam_memb
0,3,-0.592240,1,0,-0.312011,0,1,0,1
1,1,0.638529,1,0,2.461242,1,0,0,1
2,3,-0.284548,0,0,-0.282777,0,0,1,0
3,1,0.407760,1,0,1.673732,0,0,0,1
4,3,0.407760,0,0,-0.277363,0,1,1,0
...,...,...,...,...,...,...,...,...,...
886,2,-0.207624,0,0,-0.062981,0,1,1,0
887,1,-0.823009,0,0,0.673281,0,0,1,0
888,3,0.000000,1,2,0.389604,0,0,0,3
889,1,-0.284548,0,0,0.673281,1,1,1,0


In [25]:
Y

0      0
1      1
2      1
3      1
4      0
      ..
886    0
887    1
888    0
889    1
890    0
Name: Survived, Length: 891, dtype: int64

- Necessário achar os melhores parâmetros com *GridSearch* em: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html

### Regressão Logística 

In [27]:
from sklearn.linear_model import LogisticRegression

# Criando o classificador
clf_rl = LogisticRegression(random_state=42)

In [28]:
# Definindo os parâmetros
parametros_rl = {
    'penalty': ['l1','l2'],
    'C': [0.01,0.1,1,10],
    'solver': ['lbfgs','liblinear','saga'],
    'max_iter': [100,1000,5000,10000]
}

### Random Forest

In [30]:
from sklearn.ensemble import RandomForestClassifier

# Criando o classificador
clf_rf = RandomForestClassifier(random_state=42)

In [31]:
# Definindo os parâmetros
parametros_rf = {
    'n_estimators': [100,200,500,1000],
    'criterion': ['gini','entropy','log_loss'],
    'max_depth': [2,4,6,8,None],
    'max_features': ['sqrt','log2',None]
}

### MLPClassifier (Redes Neurais) 

In [32]:
from sklearn.neural_network import MLPClassifier

# Criando o classificador
clf_mlp = MLPClassifier(random_state=42)

In [34]:
# Definindo os parâmetros
parametros_mlp = {
    'solver':  ['lbfgs','sgd','adam'],
    'alpha': [10.0**(-1),10.0**(-5),10.0**(-7),10.0**(-10)],
    'max_iter': [200,500,1000,5000]
}

### Grid Search 

In [40]:
# Ignorar avisos 
import warnings
warnings.filterwarnings('ignore')

In [41]:
# Importando o datetime para visualizar tempo de execução
from datetime import datetime 
def hora_atual():
    agora = datetime.now()
    print(str(agora.hour)+':'+str(agora.minute)+":"+str(agora.second))

In [39]:
# Importando o KFold e o GridSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold

In [42]:
# Regressão Logística
hora_atual()
kfold_rl = KFold(shuffle=True,random_state=42,n_splits=8)
grid_search_rl = GridSearchCV(clf_rl, parametros_rl,scoring='accuracy',cv=kfold_rl)
grid_search_rl = grid_search_rl.fit(X_train,Y_train)
hora_atual()

12:51:29
12:51:48


In [43]:
# RandomForest
hora_atual()
kfold_rf = KFold(shuffle=True,random_state=42,n_splits=8)
grid_search_rf = GridSearchCV(clf_rf, parametros_rf,scoring='accuracy',cv=kfold_rf)
grid_search_rf = grid_search_rf.fit(X_train,Y_train)
hora_atual()

12:52:9
13:18:10


In [44]:
# MLPClassifier
hora_atual()
kfold_mlp = KFold(shuffle=True,random_state=42,n_splits=8)
grid_search_mlp = GridSearchCV(clf_mlp, parametros_mlp,scoring='accuracy',cv=kfold_mlp)
grid_search_mlp = grid_search_mlp.fit(X_train,Y_train)
hora_atual()

13:28:18
13:39:28


## Visualizando os scores

In [45]:
# Verificando o melhor score da regressão logística
grid_search_rl.best_score_

0.8039414414414414

In [46]:
# RandomForest
grid_search_rf.best_score_

0.837454954954955

In [47]:
# MLPClassifier
grid_search_mlp.best_score_

0.8123423423423424

## Visualizando os parâmetros

In [49]:
# Verificando os melhores parâmetros da regressão logística
grid_search_rl.best_params_

{'C': 0.1, 'max_iter': 100, 'penalty': 'l2', 'solver': 'saga'}

In [50]:
# RandomForest
grid_search_rf.best_params_

{'criterion': 'gini',
 'max_depth': 8,
 'max_features': 'sqrt',
 'n_estimators': 100}

In [52]:
# MLPClassifier
grid_search_mlp.best_params_

{'alpha': 0.1, 'max_iter': 500, 'solver': 'adam'}

## Previsão nos dados de validação com cada um dos melhores modelo

In [54]:
# Regressão logística
clf_best_rl = grid_search_rl.best_estimator_
Y_previsao_rl = clf_best_rl.predict(X_val)
Y_previsao_rl

array([0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0,
       0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0,
       0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0,
       0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0,
       1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0,
       0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1,
       0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0,
       0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 1, 0], dtype=int64)

In [55]:
# Para o RandomForest
clf_best_rf = grid_search_rf.best_estimator_
Y_previsao_rf = clf_best_rf.predict(X_val)
Y_previsao_rf

array([0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 0, 0, 1, 0,
       0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0,
       0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0,
       1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0,
       0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 0,
       0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 1, 0, 0, 1, 1, 0], dtype=int64)

In [56]:
# e para o MLPClassifier
clf_best_mlp = grid_search_mlp.best_estimator_
Y_previsao_mlp = clf_best_mlp.predict(X_val)
Y_previsao_mlp

array([0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1,
       0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0,
       0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1,
       0, 0, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0,
       0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0,
       1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0,
       0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0,
       0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0,
       1, 0, 1, 1, 0, 0, 1, 1, 0], dtype=int64)

## Validação dos modelos

### Avaliação da acurácia

In [57]:
from sklearn.metrics import accuracy_score

# Regressão Logística
accuracy_score(Y_val, Y_previsao_rl)

0.8169491525423729

In [59]:
# Random Forest
accuracy_score(Y_val, Y_previsao_rf)

0.8135593220338984

In [63]:
# MLPClassifier 
accuracy_score(Y_val, Y_previsao_mlp)

0.8169491525423729

### Avaliação matriz de confusão

In [61]:
from sklearn.metrics import confusion_matrix

# Regressão Logística
confusion_matrix(Y_val, Y_previsao_rl)

array([[157,  18],
       [ 36,  84]], dtype=int64)

In [62]:
# Random Forest
confusion_matrix(Y_val, Y_previsao_rf)

array([[160,  15],
       [ 40,  80]], dtype=int64)

In [64]:
# MLPClassifier 
confusion_matrix(Y_val, Y_previsao_mlp)

array([[158,  17],
       [ 37,  83]], dtype=int64)

## Previsão na base de teste

In [65]:
X_train

,Pclass,Age,SibSp,Parch,Fare,Embarked,MaleCheck,Alone,Fam_memb
6,1,1.869299,0,0,1.620136,0,1,1,0
718,3,0.000000,0,0,0.045293,2,1,1,0
685,2,-0.361471,1,2,1.174771,1,1,0,3
73,3,-0.284548,1,0,0.000000,1,1,0,1
882,3,-0.592240,0,0,-0.170531,0,0,1,0
...,...,...,...,...,...,...,...,...,...
106,3,-0.669163,0,0,-0.294687,0,0,1,0
270,1,0.000000,0,0,0.716591,0,1,1,0
860,3,0.869299,2,0,-0.014981,0,1,0,2
435,1,-1.207624,1,2,4.571140,0,0,0,3


In [66]:
teste

,PassengerId,Pclass,Age,SibSp,Parch,Fare,Embarked,MaleCheck,Alone,Fam_memb
0,892,3,0.331562,0,0,-0.280670,2,1,1,1
1,893,3,1.311954,1,0,-0.315800,0,0,0,1
2,894,2,2.488424,0,0,-0.201943,2,1,1,0
3,895,3,-0.256674,0,0,-0.245367,0,1,1,1
4,896,3,-0.648831,1,1,-0.091793,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
413,1305,3,0.000000,0,0,-0.271316,0,1,1,0
414,1306,1,0.684503,0,0,4.001229,1,0,1,0
415,1307,3,0.645287,0,0,-0.305208,0,1,1,0
416,1308,3,0.000000,0,0,-0.271316,0,1,1,2


In [ ]:
# Para a base de teste ser igual a base de treino, precisamos eliminar a coluna de id
X_teste = teste.drop('PassengerId',axis=1)

# Utilizando o melhor modelo na base de teste
Y_previsao = clf_best_rf.predict(X_teste)